In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv(
    "../Data/iris.csv",
    header=None
    )

In [4]:
df.shape

(150, 5)

In [5]:
df.head

<bound method NDFrame.head of        0    1    2    3               4
0    5.1  3.5  1.4  0.2     Iris-setosa
1    4.9  3.0  1.4  0.2     Iris-setosa
2    4.7  3.2  1.3  0.2     Iris-setosa
3    4.6  3.1  1.5  0.2     Iris-setosa
4    5.0  3.6  1.4  0.2     Iris-setosa
..   ...  ...  ...  ...             ...
145  6.7  3.0  5.2  2.3  Iris-virginica
146  6.3  2.5  5.0  1.9  Iris-virginica
147  6.5  3.0  5.2  2.0  Iris-virginica
148  6.2  3.4  5.4  2.3  Iris-virginica
149  5.9  3.0  5.1  1.8  Iris-virginica

[150 rows x 5 columns]>

In [6]:
X_test = df.iloc[:,0:4].sample(frac=0.2, random_state=42)
y_test = df.iloc[:,4:5].sample(frac=0.2, random_state=42)

X_train = df.iloc[:,0:4].drop(X_test.index)
y_train = df.iloc[:,4:5].drop(y_test.index)

X_test = X_test.to_numpy()
y_test = y_test.to_numpy()

X_train = X_train.to_numpy()
y_train = y_train.to_numpy()

In [23]:
class KNearestNeighbors:
    def __init__(self, k, distance_metric="euclidean", weights="uniform"):
        self.k = k
        self.distance_metric = distance_metric
        self.weights = weights

    def fit(self, X, y):
        """
        Store the 'prior knowledge' of the model, that will be used
        to predict new labels.
        """
        self.stored_X = X
        self.stored_y = y



    def predict(self, X):
        """
        Run the KNN classification on X
        """

        y = []

        for p in X:
            if self.distance_metric == "euclidean":
                dist = self.__euclideanDist(p, self.stored_X)
            elif self.distance_metric == "cosine":
                dist = self.__cosineDist(p, self.stored_X)
            elif self.distance_metric == "manhattan":
                dist = self.__manhattanDist(p, self.stored_X)

            best_ids = np.argsort(dist)
            best_ids = best_ids[:self.k]
            classes = self.stored_y[best_ids]

            if self.weights == "uniform":
                values, counts = np.unique(classes, return_counts=True)
                most_freq = values[np.argmax(counts)]

            elif self.weights == "distance":
                weights = 1/dist[best_ids]
                values = np.unique(classes)
                tot_weights = np.array([
                    weights[classes == c].sum()
                    for c in values
                ])

                most_freq = values[np.argmax(tot_weights)]
            
            y.append(most_freq)

        return np.array(y)


    def __euclideanDist(self, p, q):
        sq_diff = (p - q) ** 2
        point_sum = np.sum(sq_diff, axis=1)
        ed = np.sqrt(point_sum)

        return ed

    def __cosineDist(self, p, q):
        return 1 - self.__cosineSim(p, q)

    def __cosineSim(self, p, q):
        num = np.sum(p * q, axis=1)
        den = np.sqrt(np.sum(p * p)) * np.sqrt(np.sum(q * q, axis=1))

        return num/den

    def __manhattanDist(self, p, q):
        return np.sum(np.abs(p - q), axis=1)

In [24]:
knn = KNearestNeighbors(k=5)

knn.fit(X_train, y_train)

In [25]:
y_pred = knn.predict(X_test)


In [26]:
acc = np.mean(y_pred == y_test)
acc

np.float64(0.33555555555555555)

In [32]:
df2 = pd.read_csv("../Data/mnist_test.csv", header=None)

In [33]:
df2.describe()

,0,1,2,3,4,5,6,7,8,9,...,775,776,777,778,779,780,781,782,783,784
count,10000.000000,10000.0,10000.0,10000.0,10000.0,10000.0,10000.0,10000.0,10000.0,10000.0,...,10000.000000,10000.000000,10000.000000,10000.0000,10000.0,10000.0,10000.0,10000.0,10000.0,10000.0
mean,4.443400,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.179300,0.163600,0.052600,0.0006,0.0,0.0,0.0,0.0,0.0,0.0
std,2.895865,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,5.674149,5.736072,2.420004,0.0600,0.0,0.0,0.0,0.0,0.0,0.0
min,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.0000,0.0,0.0,0.0,0.0,0.0,0.0
25%,2.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.0000,0.0,0.0,0.0,0.0,0.0,0.0
50%,4.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.0000,0.0,0.0,0.0,0.0,0.0,0.0
75%,7.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.000000,0.000000,0.000000,0.0000,0.0,0.0,0.0,0.0,0.0,0.0
max,9.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,253.000000,253.000000,156.000000,6.0000,0.0,0.0,0.0,0.0,0.0,0.0


In [34]:
df2_sample = df2.groupby(0, group_keys=False).sample(n=100, random_state=42)

In [35]:
df2_sample.shape

(1000, 785)

In [37]:
X2_test = df2_sample.iloc[:,1:].sample(frac=0.2, random_state=42)
y2_test = df2_sample.iloc[:,0].sample(frac=0.2, random_state=42)

X2_train = df2_sample.iloc[:,1:].drop(X2_test.index)
y2_train = df2_sample.iloc[:,0].drop(X2_test.index)

X2_test = X2_test.to_numpy()
y2_test = y2_test.to_numpy()

X2_train = X2_train.to_numpy()
y2_train = y2_train.to_numpy()

In [38]:
knn.fit(X2_train, y2_train)

In [40]:
y2_pred = knn.predict(X2_test)


In [42]:
acc = np.mean(y2_pred == y2_test)
acc

np.float64(0.91)